In [1]:
# conda env spec

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import netCDF4 as nc
from sklearn.linear_model import LinearRegression

from pathlib import Path

PROJECT_ROOT = Path().resolve().parent.parent.parent
INTERIM_DATA = PROJECT_ROOT / 'data' / 'interim'
RAW_DATA = PROJECT_ROOT / 'data' / 'raw'
FIGURE_OUT = PROJECT_ROOT / 'figures'

savefigs = False
fig_version = 'V03'

In [2]:
def plot_box(ll, ur, f, ax, c='r'):
    ls = '--'
    ax.plot([ll[0], ur[0]], [ll[1], ll[1]], c=c, ls=ls) # lower line
    ax.plot([ll[0], ur[0]], [ur[1], ur[1]], c=c, ls=ls) # upper line
    ax.vlines([ll[0], ur[0]], ll[1], ur[1], colors=c, ls=ls) # vert lines

In [3]:
def load_misr_aerosols(aerosol_file):
    dataset = nc.Dataset(aerosol_file,'r')
    products = xr.open_dataset(xr.backends.NetCDF4DataStore(dataset['/4.4_KM_PRODUCTS/']))
    aux = xr.open_dataset(xr.backends.NetCDF4DataStore(dataset['/4.4_KM_PRODUCTS/AUXILIARY/']))
    merged = xr.merge([products, aux])
    aerosol_ds = merged.sel(X_Dim=slice(1.774e7,1.816e7), Y_Dim=slice(-0.6e6, 0.6e6)).copy(deep=True)
    aerosol_ds['aod670'] = aerosol_ds.Aerosol_Optical_Depth * (0.67 / 0.55) ** aerosol_ds.Angstrom_Exponent_550_860nm
    return aerosol_ds

In [4]:
# Load MISR OBS
obs_file = INTERIM_DATA / 'MISR_OBS_CAMP2Ex_RF17_wide.nc'
obs = xr.load_dataset(obs_file, engine='netcdf4')

# Load MISR CTH
cth_file = INTERIM_DATA / 'MISR_CTH_CAMP2Ex_RF17_wide.nc'
cth = xr.load_dataset(cth_file)

# Load P3 data
p3_file = INTERIM_DATA / 'CAMP2Ex_SPEC_RF17_preprocessed.nc'
p3 = xr.load_dataset(p3_file)
p3 = p3.sel(time=slice('2019-10-02T00:00:00', '2019-10-02T01:50:00'))

# Load MISR Aerosol data
aerosol_file = RAW_DATA / 'MISR' / 'MISR_AM1_AS_AEROSOL_P117_O105254_F13_0023.nc'
aerosol_ds = load_misr_aerosols(aerosol_file)

# Load ERA5 data
interp_time = np.datetime64('2019-10-02T02:40:00')

era_surf_file = RAW_DATA / 'era5' / 'ERA5_surface_CAMP2Ex_RF17_wide.nc'
era_surf_all = xr.load_dataset(era_surf_file)
era_surf = era_surf_all.interp(valid_time=interp_time)

era_surf_file_ens_mean = RAW_DATA / 'era5' / 'ERA5_surface_CAMP2Ex_RF17_wide_ensemble_mean.nc'
era_surf_all_ens_mean = xr.load_dataset(era_surf_file_ens_mean)
era_surf_ens_mean = era_surf_all_ens_mean.interp(valid_time=interp_time)

era_surf_file_ens_spread = RAW_DATA / 'era5' / 'ERA5_surface_CAMP2Ex_RF17_wide_ensemble_spread.nc'
era_surf_all_ens_spread = xr.load_dataset(era_surf_file_ens_spread)
era_surf_ens_spread = era_surf_all_ens_spread.interp(valid_time=interp_time)

era_upper_file = RAW_DATA / 'era5' / 'ERA5_upperAir_CAMP2Ex_RF17.nc'
era_upper_all = xr.load_dataset(era_upper_file)
era_upper = era_upper_all.interp(valid_time=interp_time)
era_1000hpa = era_upper.sel(pressure_level=1000)
era_975hpa = era_upper.sel(pressure_level=975)
era_950hpa = era_upper.sel(pressure_level=950)

# Load MISR CMV data
cmv_file = INTERIM_DATA / 'MISR_CMV_CAMP2Ex_RF17_wide.nc'
misr_cmv = xr.load_dataset(cmv_file)


In [8]:
# Set local target point
sm_obs_file = INTERIM_DATA / 'MISR_OBS_CAMP2Ex_RF17_default.nc'
sm_obs = xr.load_dataset(sm_obs_file, engine='netcdf4')
target_lat = sm_obs.center_lat.data
target_lon = sm_obs.center_lon.data
target_ll = [target_lon-0.1, target_lat-0.1]
target_ur = [target_lon+0.1, target_lat+0.1]

pristine = p3.aerosols.data == 0
trans = p3.aerosols.data == 1
polluted = p3.aerosols.data == 2

prist_color = 'tab:green'
trans_color = 'tab:gray'
poll_color = 'tab:brown'

target_color = 'fuchsia'
cth_cmap = 'terrain'
aod_cmap = 'hsv'
aerosol_lrm_cmap = 'gist_ncar'
era5_cmap = 'summer'
misr_cmv_cmap = 'tab10_r'

# lon_window = [118, 123]
# lat_window = [17.4, 21.5]
lon_window = [119.2, 121.6]
lat_window = [18.0, 20.8]
# lon_window = [119.7, 121.0]
# lat_window = [18.5, 20.2]


# Anything and Everything

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(5,5))

# Plot MISR BRF
brf_ax = ax.contourf(obs.lon275.data, obs.lat275.data, obs.brf.isel(camera=4).data, vmin=0, vmax=0.30, cmap='gray', levels=100)
# plt.colorbar(brf_ax, label='MISR 0.67 um BRF')

# Plot MISR CTH
# levels = np.arange(0, 1500, 100)
# cth_ax = ax.contourf(cth.lon1100.data, cth.lat1100.data, cth.cth.data, alpha=1.0, vmin=0, vmax=1400, levels=levels, cmap=cth_cmap)
# plt.colorbar(cth_ax, label='Cloud Top Height (m)')

# Plot MISR aerosols
# as_lon = np.ravel(aerosol_ds.Longitude.data)
# as_lat = np.ravel(aerosol_ds.Latitude.data)
# aod = ax.scatter(as_lon, as_lat, c=aerosol_ds.aod670.data, s=13, alpha=1.0, marker='s', vmin=None, vmax=None, cmap=aod_cmap)
# plt.colorbar(aod, label='670 nm aod')

# Plot MISR aerosol lowest residual mixture
# as_lon = np.ravel(aerosol_ds.Longitude.data)
# as_lat = np.ravel(aerosol_ds.Latitude.data)
# aersol_lrm_ax = ax.scatter(as_lon, as_lat, c=aerosol_ds.Lowest_Residual_Mixture.data, s=13, alpha=1.0, marker='s', vmin=None, vmax=None, cmap=aerosol_lrm_cmap)
# plt.colorbar(aersol_lrm_ax, label='Lowest Residual Mixture')

# Plot P3 Flight tracks
# p3track_size = 0.25
# ax.scatter(p3.Longitude.data[pristine], p3.Latitude.data[pristine], s=p3track_size, c=prist_color, label='Pristine')
# ax.scatter(p3.Longitude.data[trans], p3.Latitude.data[trans], s=p3track_size, c=trans_color, label='Transition')
# ax.scatter(p3.Longitude.data[polluted], p3.Latitude.data[polluted], s=p3track_size, c=poll_color, label='Polluted')

# Plot SPEC samples
# spec_re = ax.scatter(p3.Longitude.data, p3.Latitude.data, s=7, alpha=1.0, c=p3.Reff_um.data, cmap='cividis')
# plt.colorbar(spec_re, label='Reff (um)')

# Plot ERA5 wind barbs
# 10 m winds
# ax.barbs(era_surf.longitude, era_surf.latitude, era_surf.u10*1.94384, era_surf.v10*1.94384, barbcolor='tab:orange', label='ERA5 10 m winds')
# 100 m winds
# ax.barbs(era_surf.longitude, era_surf.latitude, era_surf.u100*1.94384, era_surf.v100*1.94384, barbcolor='tab:red', label='ERA5 100 m winds')
# 1000 hpa winds
# era1000hpa_ax = ax.barbs(era_1000hpa.longitude, era_1000hpa.latitude, era_1000hpa.u*1.94384, era_1000hpa.v*1.94384, era_1000hpa.z, label='ERA5 1000 hpa winds', cmap=era5_cmap)
# plt.colorbar(era1000hpa_ax, label='era5 wind barb altitude (m)')

# Plot MISR CMV wind barbs
misr_cmv_ax = ax.barbs(misr_cmv.lon17600, misr_cmv.lat17600, misr_cmv.cmv_E*1.94384, misr_cmv.cmv_N*1.94384, misr_cmv.cth_motion, clim=(0, 1000), cmap=misr_cmv_cmap)
plt.colorbar(misr_cmv_ax, label='MISR CMV altitude (m)')

# Plot AT3D retreival center & sampling points
# ax.scatter(target_lon, target_lat, marker='*', c=target_color, s=30, label='Retrieval Center')
plot_box(target_ll, target_ur, f, ax, c='r')
# ax.scatter(obs.lon17600.data, obs.lat17600, s=3, c='tab:red')

# Adjust window
ax.set_xlim(lon_window)
ax.set_ylim(lat_window)
ax.grid()
# ax.legend(loc='upper left')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

## Cloud Top Height

In [ ]:
lcl = 400

In [ ]:
f, ax = plt.subplots(1, 2, figsize=(15,6))

# Plot MISR BRF
ax[0].contourf(obs.lon275.data, obs.lat275.data, obs.brf.isel(camera=4).data, vmin=0, vmax=0.20, cmap='gray', levels=100)
ax[0].set_xlim(lon_window)
ax[0].set_ylim(lat_window)

# Plot MISR CTH
levels = np.arange(0, 1500, 100)
cth_ax = ax[0].contourf(cth.lon1100.data, cth.lat1100.data, cth.cth.data, alpha=1.0, vmin=0, vmax=1400, levels=levels, cmap=cth_cmap)
plt.colorbar(cth_ax, ax=ax[0], label='Cloud Top Height (m)')

# Plot AT3D retreival center & sampling points
ax[0].scatter(target_lon, target_lat, marker='*', c=target_color, s=30, label='Retrieval Center')
ax[0].scatter(obs.lon17600.data, obs.lat17600, s=3, c='tab:red')

bins = np.arange(-1, 1501, 10) + 0.5
h = ax[1].hist(np.ravel(cth.cth.data), bins=bins)
ax[1].vlines(lcl, 0, np.max(h[0]), colors='tab:orange')
ax[1].set_xlabel('Cloud Top Height (m)')
ax[1].set_ylabel('Count')
ax[1].set_title('Whole Scene')
# print(h[1])

In [ ]:
# Subset conditions
lon_range = 0.1
lat_range = 0.1
ll = [target_lon-lon_range, target_lat-lat_range]
ur = [target_lon+lon_range, target_lat+lat_range]
lon_cond = (cth.lon1100.data > ll[0]) & (cth.lon1100.data < ur[0])
lat_cond = (cth.lat1100.data > ll[1]) & (cth.lat1100.data < ur[1])
sub_cond = lon_cond & lat_cond

lon_pad = 1.5
lat_pad = 1.5

In [ ]:
f, ax = plt.subplots(1, 2, figsize=(15,6))

# Plot MISR BRF
ax[0].contourf(obs.lon275.data, obs.lat275.data, obs.brf.isel(camera=4).data, vmin=0, vmax=0.20, cmap='gray', levels=100)
ax[0].set_xlim([ll[0] - lon_pad*lon_range, ur[0] + lon_pad*lon_range])
ax[0].set_ylim([ll[1] - lat_pad*lat_range, ur[1] + lat_pad*lat_range])

# Plot MISR CTH
levels = np.arange(0, 1500, 100)
cth_ax = ax[0].contourf(cth.lon1100.data, cth.lat1100.data, cth.cth.data, alpha=1.0, vmin=0, vmax=1400, levels=levels, cmap=cth_cmap)
plt.colorbar(cth_ax, ax=ax[0], label='Cloud Top Height (m)')

plot_box(ll, ur, f, ax[0], c='r')

# Plot AT3D retreival center & sampling points
ax[0].scatter(target_lon, target_lat, marker='*', c=target_color, s=30, label='Retrieval Center')
ax[0].scatter(obs.lon17600.data, obs.lat17600, s=3, c='tab:red')

bins = np.arange(-1, 1501, 10) + 0.5
h = ax[1].hist(np.ravel(cth.cth.data[sub_cond]), bins=bins)
ax[1].vlines(lcl, 0, np.max(h[0]), colors='tab:orange')
ax[1].set_xlabel('Cloud Top Height (m)')
ax[1].set_ylabel('Count')
ax[1].set_title('Sub Scene')
# print(h[1])

## Surface Winds

In [9]:
era_surf_point = era_surf.interp(latitude=target_lat, longitude=target_lon, method='linear')
print('10 meter winds at target({:.3f}, {:.3f}):'.format(target_lat, target_lon))
print('   u  = {:.3f} m/s'.format(era_surf_point.u10.data))
print('   v  = {:.3f} m/s'.format(era_surf_point.v10.data))
print('  |s| = {:.3f} m/s'.format((era_surf_point.u10.data**2 + era_surf_point.v10.data**2)**0.5))

print('100 meter winds at target ({:.3f}, {:.3f}):'.format(target_lat, target_lon))
print('   u  = {:.3f} m/s'.format(era_surf_point.u100.data))
print('   v  = {:.3f} m/s'.format(era_surf_point.v100.data))
print('  |s| = {:.3f} m/s'.format((era_surf_point.u100.data**2 + era_surf_point.v100.data**2)**0.5))

10 meter winds at target(19.266, 119.724):
   u  = 0.557 m/s
   v  = 0.373 m/s
  |s| = 0.670 m/s
100 meter winds at target (19.266, 119.724):
   u  = 0.570 m/s
   v  = 0.382 m/s
  |s| = 0.686 m/s


In [12]:
era_surf_ens_mean_point = era_surf_ens_mean.interp(latitude=target_lat, longitude=target_lon, method='linear')
era_surf_ens_spread_point = era_surf_ens_spread.interp(latitude=target_lat, longitude=target_lon, method='linear')
print('10 meter ensemble mean +/- spread at target({:.3f}, {:.3f}):'.format(target_lat, target_lon))
print('   u  = {:.3f} +/- {:.3f} m/s'.format(era_surf_ens_mean_point.u10.data, era_surf_ens_spread_point.u10.data))
print('   v  = {:.3f} +/- {:.3f} m/s'.format(era_surf_ens_mean_point.v10.data, era_surf_ens_spread_point.v10.data))
print('  |s| = {:.3f} m/s'.format((era_surf_ens_mean_point.u10.data**2 + era_surf_ens_mean_point.v10.data**2)**0.5))

10 meter ensemble mean +/- spread at target(19.266, 119.724):
   u  = 0.932 +/- 0.246 m/s
   v  = -0.213 +/- 0.322 m/s
  |s| = 0.956 m/s


In [14]:
era_surf_point = era_surf.interp(latitude=target_lat, longitude=target_lon, method='linear')
u10 = era_surf_point.u10.data
sigma_u10 = era_surf_ens_spread_point.u10.data
v10 = era_surf_point.v10.data
sigma_v10 = era_surf_ens_spread_point.v10.data
s10 = (u10**2 + v10**2)**0.5
sigma_s10 = ((u10**2 * sigma_u10**2 + v10**2 * sigma_v10**2)**0.5) / s10
print('10 meter winds at target({:.3f}, {:.3f}):'.format(target_lat, target_lon))
print('   u  = {:.3f} +/- {:.3f} m/s'.format(u10, sigma_u10))
print('   v  = {:.3f} +/- {:.3f} m/s'.format(v10, sigma_v10))
print('  |s| = {:.3f} +/- {:.3f} m/s'.format(s10, sigma_s10))

10 meter winds at target(19.266, 119.724):
   u  = 0.557 +/- 0.246 m/s
   v  = 0.373 +/- 0.322 m/s
  |s| = 0.670 +/- 0.272 m/s


## Upper level winds

### MISR Cloud Motion Vectors

In [ ]:
target_cond = (misr_cmv.lat17600.data == target_lat) & (misr_cmv.lon17600.data == target_lon)
assert np.sum(target_cond == 1)
target_cth_motion = misr_cmv.cth_motion.data[target_cond][0]
target_cmv_E = misr_cmv.cmv_E.data[target_cond][0]
target_cmv_N = misr_cmv.cmv_N.data[target_cond][0]

print('MISR CMV at target ({:.3f}, {:.3f})'.format(target_lat, target_lon))
print('   Height of Motion = {:.3f} m'.format(target_cth_motion))
print('   CMV East = {:.3f}'.format(target_cmv_E))
print('   CMV North = {:.3f}'.format(target_cmv_N))
print('   CMV Speed = {:.3f}'.format((target_cmv_E**2 + target_cmv_N**2)**0.5))
print('   CMV Direction = {:.3f} degrees from north going counterclockwise'.format(np.degrees(np.arctan2(target_cmv_E, target_cmv_N))))

In [ ]:
# lon_range = 0.53
# lat_range = 0.6
lon_range = 0.2
lat_range = 0.25
ll = [target_lon-lon_range, target_lat-lat_range]
ur = [target_lon+lon_range, target_lat+lat_range]
lon_cond = (misr_cmv.lon17600.data > ll[0]) & (misr_cmv.lon17600.data < ur[0])
lat_cond = (misr_cmv.lat17600.data > ll[1]) & (misr_cmv.lat17600.data < ur[1])
sub_cond = lon_cond & lat_cond

lon_pad = 2.0
lat_pad = 2.0

bin_size = 100
cth_window = [target_cth_motion - 0.5*bin_size, target_cth_motion + 0.5*bin_size]
cth_cond = (misr_cmv.cth_motion.data > cth_window[0]) & (misr_cmv.cth_motion.data < cth_window[1]) & lat_cond & lon_cond
not_cth_cond = np.logical_not(cth_cond)

plt.style.use('seaborn-v0_8-paper')
f, ax = plt.subplots(1, 1, figsize=(6,5))

# Plot MISR BRF
ax.contourf(obs.lon275.data, obs.lat275.data, obs.brf.isel(camera=4).data, vmin=0, vmax=0.20, cmap='gray', levels=100)

# Plot MISR CMV quivers
scale = 20
misr_cmv_ax_bold = ax.quiver(
    misr_cmv.lon17600.data[cth_cond],
    misr_cmv.lat17600.data[cth_cond],
    misr_cmv.cmv_E.data[cth_cond],
    misr_cmv.cmv_N.data[cth_cond],
    misr_cmv.cth_motion.data[cth_cond],
    units='width',
    scale=scale,
    width=0.015,
    headwidth = 3,
    headaxislength = 3,
    headlength = 3,
    cmap='hsv',
    clim=(0,1000)
)
misr_cmv_ax = ax.quiver(
    misr_cmv.lon17600.data[not_cth_cond],
    misr_cmv.lat17600.data[not_cth_cond], 
    misr_cmv.cmv_E.data[not_cth_cond],
    misr_cmv.cmv_N.data[not_cth_cond],
    misr_cmv.cth_motion.data[not_cth_cond],
    units='width',
    scale=scale,
    width=0.005,
    headwidth = 3,
    headaxislength = 3,
    headlength = 3,
    cmap='hsv',
    clim=(0,1000)
)
ax.quiverkey(misr_cmv_ax_bold, 0.8, 0.04, 5, '5 m/s', labelpos='N', coordinates='figure')
# ax.quiverkey(misr_cmv_ax, 0.8, 0.02, 5, '5 m/s', labelpos='N', coordinates='figure')

plt.colorbar(misr_cmv_ax, label='MISR CMV altitude (m)')

# Plot AT3D retreival center & sampling points
# ax.scatter(target_lon, target_lat, marker='*', c=target_color, s=30, label='Retrieval Center')
# ax.scatter(obs.lon17600.data, obs.lat17600, s=3, c='tab:red')

# plot_box(ll, ur, f, ax, c='r')
plot_box(target_ll, target_ur, f, ax, c='r')

# Adjust window
ax.set_xlim([ll[0] - lon_pad*lon_range, ur[0] + lon_pad*lon_range])
ax.set_ylim([ll[1] - lat_pad*lat_range, ur[1] + lat_pad*lat_range])
# ax.set_xlim(lon_window)
# ax.set_ylim(lat_window)
ax.grid()
# ax.legend(loc='upper left')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

print('Average of n = {} CMVs between {:.1f} m and {:.1f} m:'.format(np.sum(cth_cond), cth_window[0], cth_window[1]))
mean_E = np.nanmean(misr_cmv.cmv_E.data[cth_cond])
mean_N = np.nanmean(misr_cmv.cmv_N.data[cth_cond])
print('   Average CMV East = {:.2f} m/s'.format(mean_E))
print('   Average CMV North = {:.2f} m/s'.format(mean_N))
print('   (Maen_CMV_E**2 + Mean_CMV_N**2)**0.5 = {:.2f} m/s'.format((mean_E**2 + mean_N**2)**0.5))
print('   Average CMV Speed = {:.2f} m/s'.format(np.nanmean((misr_cmv.cmv_E.data[cth_cond]**2 + misr_cmv.cmv_N.data[cth_cond]**2)**0.5)))

## Aerosol

In [ ]:
f = plt.figure(figsize=(13,10))

gs = GridSpec(2, 2, figure=f)
ax00 = f.add_subplot(gs[0,0])
ax01 = f.add_subplot(gs[0,1])
ax1 = f.add_subplot(gs[1,:])

# Plot MISR BRF
ax00.contourf(obs.lon275.data, obs.lat275.data, obs.brf.isel(camera=4).data, vmin=0, vmax=0.20, cmap='gray', levels=100)
ax01.contourf(obs.lon275.data, obs.lat275.data, obs.brf.isel(camera=4).data, vmin=0, vmax=0.20, cmap='gray', levels=100)

# Plot MISR aerosols
as_lon = np.ravel(aerosol_ds.Longitude.data)
as_lat = np.ravel(aerosol_ds.Latitude.data)
aod = ax00.scatter(as_lon, as_lat, c=aerosol_ds.aod670.data, s=13, alpha=1.0, marker='s', vmin=None, vmax=None, cmap=aod_cmap)
plt.colorbar(aod, label='670 nm aod')
plot_box(target_ll, target_ur, f, ax00, c='r')

# Plot MISR aerosol lowest residual mixture
as_lon = np.ravel(aerosol_ds.Longitude.data)
as_lat = np.ravel(aerosol_ds.Latitude.data)
aersol_lrm_ax = ax01.scatter(as_lon, as_lat, c=aerosol_ds.Lowest_Residual_Mixture.data, s=13, alpha=1.0, marker='s', vmin=None, vmax=None, cmap=aerosol_lrm_cmap)
plt.colorbar(aersol_lrm_ax, label='Lowest Residual Mixture')
plot_box(target_ll, target_ur, f, ax01, c='r')

# Aerosol lowest residual mixture bars
n_mixtures = 12
hist, bin_edges = np.histogram(aerosol_ds.Lowest_Residual_Mixture.data, bins=74, range=(0.5, 74.5))
midpoints = (bin_edges[:-1] + bin_edges[1:])/2
sorted_hist = np.sort(hist)
sorted_hist = sorted_hist[::-1]
sorted_midpoints = midpoints[np.argsort(hist)]
sorted_midpoints = sorted_midpoints[::-1]
ax1.bar(sorted_midpoints.astype('str')[:n_mixtures], sorted_hist[:n_mixtures])
ax1.set_xlabel('Aerosol Mixture')
ax1.set_ylabel('Counts')
ax1.set_title('Lowest Residual Mixture')

# Plot AT3D retreival center & sampling points
# ax00.scatter(target_lon, target_lat, marker='*', c=target_color, s=30, label='Retrieval Center')
# ax01.scatter(target_lon, target_lat, marker='*', c=target_color, s=30, label='Retrieval Center')
plot_box(target_ll, target_ur, f, ax00, c='r')
plot_box(target_ll, target_ur, f, ax01, c='r')
# ax00.scatter(obs.lon17600.data, obs.lat17600, s=3, c='tab:red')
# ax01.scatter(obs.lon17600.data, obs.lat17600, s=3, c='tab:red')

# Adjust window
ax00.set_xlim(lon_window)
ax00.set_ylim(lat_window)
ax00.grid()
# ax00.legend(loc='upper left')
ax00.set_xlabel('Longitude')
ax00.set_ylabel('Latitude')
ax01.set_xlim(lon_window)
ax01.set_ylim(lat_window)
ax01.grid()
# ax01.legend(loc='upper left')
ax01.set_xlabel('Longitude')
ax01.set_ylabel('Latitude')

In [ ]:
# lon_range = 0.53
# lat_range = 0.6
lon_range = 0.3
lat_range = 0.3
ll = [target_lon-lon_range, target_lat-lat_range]
ur = [target_lon+lon_range, target_lat+lat_range]
lon_cond = (aerosol_ds.Longitude.data > ll[0]) & (aerosol_ds.Longitude.data < ur[0])
lat_cond = (aerosol_ds.Latitude.data > ll[1]) & (aerosol_ds.Latitude.data < ur[1])
sub_cond = lon_cond & lat_cond

lon_pad = 2.0
lat_pad = 2.0

f = plt.figure(figsize=(13,10))

gs = GridSpec(2, 2, figure=f)
ax00 = f.add_subplot(gs[0,0])
ax01 = f.add_subplot(gs[0,1])
ax1 = f.add_subplot(gs[1,:])

# Plot MISR BRF
ax00.contourf(obs.lon275.data, obs.lat275.data, obs.brf.isel(camera=4).data, vmin=0, vmax=0.20, cmap='gray', levels=100)
ax01.contourf(obs.lon275.data, obs.lat275.data, obs.brf.isel(camera=4).data, vmin=0, vmax=0.20, cmap='gray', levels=100)

# Plot MISR aerosols
as_lon = np.ravel(aerosol_ds.Longitude.data)
as_lat = np.ravel(aerosol_ds.Latitude.data)
aod = ax00.scatter(as_lon, as_lat, c=aerosol_ds.aod670.data, s=13, alpha=1.0, marker='s', vmin=None, vmax=None, cmap=aod_cmap)
plt.colorbar(aod, label='670 nm aod')

# Plot MISR aerosol lowest residual mixture
as_lon = np.ravel(aerosol_ds.Longitude.data)
as_lat = np.ravel(aerosol_ds.Latitude.data)
aersol_lrm_ax = ax01.scatter(as_lon, as_lat, c=aerosol_ds.Lowest_Residual_Mixture.data, s=13, alpha=1.0, marker='s', vmin=None, vmax=None, cmap=aerosol_lrm_cmap)
plt.colorbar(aersol_lrm_ax, label='Lowest Residual Mixture')
plot_box(ll, ur, f, ax01, c='b')
# ax01.set_xlim(lon_window)
# ax01.set_ylim(lat_window)
ax01.set_xlim([ll[0] - lon_pad*lon_range, ur[0] + lon_pad*lon_range])
ax01.set_ylim([ll[1] - lat_pad*lat_range, ur[1] + lat_pad*lat_range])
ax01.grid()
ax01.set_xlabel('Longitude')
ax01.set_ylabel('Latitude')

# Aerosol lowest residual mixture bars
n_mixtures = 9
hist, bin_edges = np.histogram(aerosol_ds.Lowest_Residual_Mixture.data[sub_cond], bins=74, range=(0.5, 74.5))
midpoints = (bin_edges[:-1] + bin_edges[1:])/2
sorted_hist = np.sort(hist)
sorted_hist = sorted_hist[::-1]
sorted_midpoints = midpoints[np.argsort(hist)]
sorted_midpoints = sorted_midpoints[::-1]
ax1.bar(sorted_midpoints.astype('str')[:n_mixtures], sorted_hist[:n_mixtures])
ax1.set_xlabel('Aerosol Mixture')
ax1.set_ylabel('Counts')
ax1.set_title('Lowest Residual Mixture')

# Plot AT3D retreival center & sampling points
# ax00.scatter(target_lon, target_lat, marker='*', c=target_color, s=30, label='Retrieval Center')
# ax01.scatter(target_lon, target_lat, marker='*', c=target_color, s=30, label='Retrieval Center')
plot_box(target_ll, target_ur, f, ax00, c='r')
plot_box(target_ll, target_ur, f, ax01, c='r')
# ax00.scatter(obs.lon17600.data, obs.lat17600, s=3, c='tab:red')
# ax01.scatter(obs.lon17600.data, obs.lat17600, s=3, c='tab:red')

# Adjust window
ax00.set_xlim(lon_window)
ax00.set_ylim(lat_window)
ax00.grid()
# ax00.legend(loc='upper left')
# ax01.legend(loc='upper left')
ax00.set_xlabel('Longitude')
ax00.set_ylabel('Latitude')

## P3

In [ ]:
# Plot P3 altitude vs time
f, ax = plt.subplots(1, 1, figsize=(7,5))
size = 8
ax.scatter(p3.time.data[pristine], p3.GPS_Altitude.data[pristine], s=size, c=prist_color, label='Pristine')
ax.scatter(p3.time.data[trans], p3.GPS_Altitude.data[trans], s=size, c=trans_color, label='Transition')
ax.scatter(p3.time.data[polluted], p3.GPS_Altitude.data[polluted], s=size, c=poll_color, label='Polluted')
p3_cond = p3.Reff_um.data >= 0
p3_reff = ax.scatter(p3.time.data[p3_cond], p3.GPS_Altitude.data[p3_cond], s=size+11, c='k', marker='x', label='Microphysics Samples')
# plt.colorbar(p3_reff, ax=ax, label='Reff (um)')
ax.legend()
ax.grid()
ax.set_xlabel('time (UTC)')
ax.set_ylabel('altitude (m)')

In [ ]:
def ReffLinearRegression(p3):
    # Prepare the data
    y = p3['Reff_um'].data
    cloud_cond = ~np.isnan(y)
    y_cloud = y[cloud_cond]
    y_pris = y[cloud_cond & pristine]
    y_poll = y[cloud_cond & polluted]
    X = p3['GPS_Altitude'].data / 1000
    X_cloud = X[cloud_cond].reshape(-1, 1)
    X_pris = X[cloud_cond & pristine].reshape(-1, 1)
    X_poll = X[cloud_cond & polluted].reshape(-1, 1)

    # Create an instance of the LinearRegression model
    m_cloud = LinearRegression()
    m_pris = LinearRegression()
    m_poll = LinearRegression()

    # Fit the model to the data
    m_cloud.fit(X_cloud, y_cloud)
    m_pris.fit(X_pris, y_pris)
    m_poll.fit(X_poll, y_poll)

    # Print the regression results
    print('All Cloud:')
    print("   Intercept:", m_cloud.intercept_)
    print("   Coefficient:", m_cloud.coef_)
    print('Pristine:')
    print("   Intercept:", m_pris.intercept_)
    print("   Coefficient:", m_pris.coef_)
    print('All Cloud:')
    print("   Intercept:", m_poll.intercept_)
    print("   Coefficient:", m_poll.coef_)

    return m_cloud, m_pris, m_poll

In [ ]:
m_cloud, m_pris, m_poll = ReffLinearRegression(p3)

# Plot effective radius vs altitude
f, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.scatter(p3.Reff_um.data[pristine], p3.GPS_Altitude[pristine], c=prist_color, label='pristine')
ax.scatter(p3.Reff_um.data[trans], p3.GPS_Altitude[trans], c=trans_color, label='trans')
ax.scatter(p3.Reff_um.data[polluted], p3.GPS_Altitude[polluted], c=poll_color, label='polluted')

z = np.linspace(400, 1600, 200)
re_cloud = m_cloud.intercept_ + m_cloud.coef_ * z/1000
re_pris = m_pris.intercept_ + m_pris.coef_ * z/1000
re_poll = m_poll.intercept_ + m_poll.coef_ * z/1000

ax.plot(re_cloud, z, color='black', label='All Cloud')
ax.plot(re_pris, z, color=prist_color, label='Pristine')
ax.plot(re_poll, z, color=poll_color, label='Polluted')
ax.set_xlabel('Reff (um)')
ax.set_ylabel('Altitude (m)')
ax.grid()
ax.legend()


In [ ]:
# Plot extinction vs altitude
f, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.scatter(p3.ext_km.data[pristine], p3.GPS_Altitude[pristine], c=prist_color, label='pristine')
ax.scatter(p3.ext_km.data[trans], p3.GPS_Altitude[trans], c=trans_color, label='trans')
ax.scatter(p3.ext_km.data[polluted], p3.GPS_Altitude[polluted], c=poll_color, label='polluted')

ax.set_xlabel('Extinction (1/km)')
ax.set_ylabel('Altitude (m)')
ax.grid()
ax.legend()
ax.set_xscale('log')


# Figures

In [ ]:
def set_labels(ax_flat):
    labels = ['a)', 'b)', 'c)', 'd)', 'e)', 'f)', 'g)', 'h)', 'i)']
    for i, a in enumerate(ax_flat):
        a.set_title(
            labels[i],
            fontfamily='serif',
            loc='left',
            fontsize='medium',
            fontweight='bold'
        )


## Figure 1: P3 Flight Tracks

In [ ]:
f, ax = plt.subplots(3, 1, figsize=(5.5,12.5))

# Plot MISR BRF
brf_ax = ax[0].contourf(obs.lon275.data, obs.lat275.data, obs.brf.isel(camera=4).data, vmin=0, vmax=0.30, cmap='gray', levels=100)
plt.colorbar(brf_ax, label='MISR 0.67 um BRF')

# Plot P3 Flight tracks
p3track_size = 0.25
ax[0].scatter(p3.Longitude.data[pristine], p3.Latitude.data[pristine], s=p3track_size, c=prist_color, label='Pristine')
ax[0].scatter(p3.Longitude.data[trans], p3.Latitude.data[trans], s=p3track_size, c=trans_color, label='Transition')
ax[0].scatter(p3.Longitude.data[polluted], p3.Latitude.data[polluted], s=p3track_size, c=poll_color, label='Polluted')

# Plot AT3D retreival center & sampling points
# ax.scatter(target_lon, target_lat, marker='*', c=target_color, s=30, label='Retrieval Center')
plot_box(target_ll, target_ur, f, ax[0], c='r')
# ax.scatter(obs.lon17600.data, obs.lat17600, s=3, c='tab:red')

# Adjust window
ax[0].set_xlim(lon_window)
ax[0].set_ylim(lat_window)
ax[0].grid()
# ax.legend(loc='upper left')
ax[0].set_xlabel('Longitude')
ax[0].set_ylabel('Latitude')

# Plot P3 altitude vs time
size = 8
ax[1].scatter(p3.time.data[pristine], p3.GPS_Altitude.data[pristine], s=size, c=prist_color, label='Pristine')
ax[1].scatter(p3.time.data[trans], p3.GPS_Altitude.data[trans], s=size, c=trans_color, label='Transition')
ax[1].scatter(p3.time.data[polluted], p3.GPS_Altitude.data[polluted], s=size, c=poll_color, label='Polluted')
p3_cond = p3.Reff_um.data >= 0
p3_reff = ax[1].scatter(p3.time.data[p3_cond], p3.GPS_Altitude.data[p3_cond], s=size+11, c='k', marker='x', label='Microphysics Samples')
# plt.colorbar(p3_reff, ax=ax[1], label='Reff (um)')
ax[1].legend()
ax[1].grid()
ax[1].set_xlabel('time (UTC)')
ax[1].set_ylabel('altitude (m)')

m_cloud, m_pris, m_poll = ReffLinearRegression(p3)

# Plot effective radius vs altitude
ax[2].scatter(p3.Reff_um.data[pristine], p3.GPS_Altitude[pristine], c=prist_color, label='pristine')
ax[2].scatter(p3.Reff_um.data[trans], p3.GPS_Altitude[trans], c=trans_color, label='trans')
ax[2].scatter(p3.Reff_um.data[polluted], p3.GPS_Altitude[polluted], c=poll_color, label='polluted')

z = np.linspace(400, 1600, 200)
re_cloud = m_cloud.intercept_ + m_cloud.coef_ * z/1000
re_pris = m_pris.intercept_ + m_pris.coef_ * z/1000
re_poll = m_poll.intercept_ + m_poll.coef_ * z/1000

ax[2].plot(re_cloud, z, color='black', label='All Cloud')
ax[2].plot(re_pris, z, color=prist_color, label='Pristine')
ax[2].plot(re_poll, z, color=poll_color, label='Polluted')
ax[2].set_xlabel('Reff (um)')
ax[2].set_ylabel('Altitude (m)')
ax[2].grid()
ax[2].legend()

set_labels(ax.flatten())

if(savefigs):
    plt.savefig(
        FIGURE_OUT / '{}_figure01_P3_tracks.png'.format(fig_version),
        format='png',
        dpi=300,
        bbox_inches='tight',
        transparent=False
    )

## Figure 3: Lower and Upper level winds

In [ ]:
# lon_range = 0.53
# lat_range = 0.6
lon_range = 0.2
lat_range = 0.25
ll = [target_lon-lon_range, target_lat-lat_range]
ur = [target_lon+lon_range, target_lat+lat_range]
lon_cond = (misr_cmv.lon17600.data > ll[0]) & (misr_cmv.lon17600.data < ur[0])
lat_cond = (misr_cmv.lat17600.data > ll[1]) & (misr_cmv.lat17600.data < ur[1])
sub_cond = lon_cond & lat_cond

lon_pad = 2.0
lat_pad = 2.0

bin_size = 100
cth_window = [target_cth_motion - 0.5*bin_size, target_cth_motion + 0.5*bin_size]
cth_cond = (misr_cmv.cth_motion.data > cth_window[0]) & (misr_cmv.cth_motion.data < cth_window[1]) & lat_cond & lon_cond
not_cth_cond = np.logical_not(cth_cond)

plt.style.use('seaborn-v0_8-paper')
f, ax = plt.subplots(1, 2, figsize=(12,5))

# Plot MISR BRF
brf_ax = ax[0].contourf(obs.lon275.data, obs.lat275.data, obs.brf.isel(camera=4).data, vmin=0, vmax=0.30, cmap='gray', levels=100)
plt.colorbar(brf_ax, label='MISR 0.67 um BRF')

scale = 20
era5_surf_wind = ax[0].quiver(
    era_surf.longitude,
    era_surf.latitude,
    era_surf.u10,
    era_surf.v10,
    color='tab:orange',
    units='width',
    scale=scale,
    width=0.01,
    headwidth = 3,
    headaxislength = 3,
    headlength = 3,
)
# ax[0].quiverkey(era5_surf_wind, 0.4, 0.04, 5, '5 m/s', labelpos='N', coordinates='figure')

# Plot AT3D retreival center & sampling points
# ax.scatter(target_lon, target_lat, marker='*', c=target_color, s=30, label='Retrieval Center')
plot_box(target_ll, target_ur, f, ax[0], c='r')
# ax.scatter(obs.lon17600.data, obs.lat17600, s=3, c='tab:red')

# Adjust window
ax[0].set_xlim([ll[0] - lon_pad*lon_range, ur[0] + lon_pad*lon_range])
ax[0].set_ylim([ll[1] - lat_pad*lat_range, ur[1] + lat_pad*lat_range])
# ax[0].set_xlim(lon_window)
# ax[0].set_ylim(lat_window)
ax[0].grid()
# ax.legend(loc='upper left')
ax[0].set_xlabel('Longitude')
ax[0].set_ylabel('Latitude')


# Plot MISR BRF
ax[1].contourf(obs.lon275.data, obs.lat275.data, obs.brf.isel(camera=4).data, vmin=0, vmax=0.20, cmap='gray', levels=100)

# Plot MISR CMV quivers
scale = 20
misr_cmv_ax_bold = ax[1].quiver(
    misr_cmv.lon17600.data[cth_cond],
    misr_cmv.lat17600.data[cth_cond],
    misr_cmv.cmv_E.data[cth_cond],
    misr_cmv.cmv_N.data[cth_cond],
    misr_cmv.cth_motion.data[cth_cond],
    units='width',
    scale=scale,
    width=0.015,
    headwidth = 3,
    headaxislength = 3,
    headlength = 3,
    cmap='hsv',
    clim=(400,1000)
)
misr_cmv_ax = ax[1].quiver(
    misr_cmv.lon17600.data[not_cth_cond],
    misr_cmv.lat17600.data[not_cth_cond], 
    misr_cmv.cmv_E.data[not_cth_cond],
    misr_cmv.cmv_N.data[not_cth_cond],
    misr_cmv.cth_motion.data[not_cth_cond],
    units='width',
    scale=scale,
    width=0.005,
    headwidth = 3,
    headaxislength = 3,
    headlength = 3,
    cmap='hsv',
    clim=(400,1000)
)
ax[1].quiverkey(misr_cmv_ax_bold, 0.5, 0.03, 5, '5 m/s', labelpos='N', coordinates='figure')
# ax.quiverkey(misr_cmv_ax, 0.8, 0.02, 5, '5 m/s', labelpos='N', coordinates='figure')

plt.colorbar(misr_cmv_ax, label='MISR CMV altitude (m)')

# Plot AT3D retreival center & sampling points
# ax.scatter(target_lon, target_lat, marker='*', c=target_color, s=30, label='Retrieval Center')
# ax.scatter(obs.lon17600.data, obs.lat17600, s=3, c='tab:red')

# plot_box(ll, ur, f, ax, c='r')
plot_box(target_ll, target_ur, f, ax[1], c='r')

# Adjust window
ax[1].set_xlim([ll[0] - lon_pad*lon_range, ur[0] + lon_pad*lon_range])
ax[1].set_ylim([ll[1] - lat_pad*lat_range, ur[1] + lat_pad*lat_range])
# ax.set_xlim(lon_window)
# ax.set_ylim(lat_window)
ax[1].grid()
# ax.legend(loc='upper left')
ax[1].set_xlabel('Longitude')
ax[1].set_ylabel('Latitude')

print('Average of n = {} CMVs between {:.1f} m and {:.1f} m:'.format(np.sum(cth_cond), cth_window[0], cth_window[1]))
mean_E = np.nanmean(misr_cmv.cmv_E.data[cth_cond])
mean_N = np.nanmean(misr_cmv.cmv_N.data[cth_cond])
print('   Average CMV East = {:.2f} m/s'.format(mean_E))
print('   Average CMV North = {:.2f} m/s'.format(mean_N))
print('   (Maen_CMV_E**2 + Mean_CMV_N**2)**0.5 = {:.2f} m/s'.format((mean_E**2 + mean_N**2)**0.5))
print('   Average CMV Speed = {:.2f} m/s'.format(np.nanmean((misr_cmv.cmv_E.data[cth_cond]**2 + misr_cmv.cmv_N.data[cth_cond]**2)**0.5)))

set_labels(ax.flatten())
ax[0].set_title('ERA5 10 m winds')
ax[1].set_title('MISR Cloud Motion Vectors')

if(savefigs):
    plt.savefig(
        FIGURE_OUT / '{}_figure03_winds.png'.format(fig_version),
        format='png',
        dpi=300,
        bbox_inches='tight',
        transparent=False
    )


## Figure 5: MISR Aerosol

In [ ]:
# lon_range = 0.53
# lat_range = 0.6
lon_range = 0.3
lat_range = 0.3
ll = [target_lon-lon_range, target_lat-lat_range]
ur = [target_lon+lon_range, target_lat+lat_range]
lon_cond = (aerosol_ds.Longitude.data > ll[0]) & (aerosol_ds.Longitude.data < ur[0])
lat_cond = (aerosol_ds.Latitude.data > ll[1]) & (aerosol_ds.Latitude.data < ur[1])
sub_cond = lon_cond & lat_cond

lon_pad = 2.0
lat_pad = 2.0

plt.style.use('seaborn-v0_8-paper')
f = plt.figure(figsize=(13,10))

gs = GridSpec(2, 2, figure=f)
ax00 = f.add_subplot(gs[0,0])
ax01 = f.add_subplot(gs[0,1])
ax1 = f.add_subplot(gs[1,:])

# Plot MISR BRF
ax00.contourf(obs.lon275.data, obs.lat275.data, obs.brf.isel(camera=4).data, vmin=0, vmax=0.20, cmap='gray', levels=100)
ax01.contourf(obs.lon275.data, obs.lat275.data, obs.brf.isel(camera=4).data, vmin=0, vmax=0.20, cmap='gray', levels=100)

# Plot MISR aerosols
as_lon = np.ravel(aerosol_ds.Longitude.data)
as_lat = np.ravel(aerosol_ds.Latitude.data)
aod = ax00.scatter(as_lon, as_lat, c=aerosol_ds.aod670.data, s=13, alpha=1.0, marker='s', vmin=None, vmax=None, cmap=aod_cmap)
plt.colorbar(aod, label='670 nm aod')

# Plot MISR aerosol lowest residual mixture
as_lon = np.ravel(aerosol_ds.Longitude.data)
as_lat = np.ravel(aerosol_ds.Latitude.data)
aersol_lrm_ax = ax01.scatter(as_lon, as_lat, c=aerosol_ds.Lowest_Residual_Mixture.data, s=13, alpha=1.0, marker='s', vmin=None, vmax=None, cmap=aerosol_lrm_cmap)
plt.colorbar(aersol_lrm_ax, label='Lowest Residual Mixture')
plot_box(ll, ur, f, ax01, c='b')
# ax01.set_xlim(lon_window)
# ax01.set_ylim(lat_window)
ax01.set_xlim([ll[0] - lon_pad*lon_range, ur[0] + lon_pad*lon_range])
ax01.set_ylim([ll[1] - lat_pad*lat_range, ur[1] + lat_pad*lat_range])
ax01.grid()
ax01.set_xlabel('Longitude')
ax01.set_ylabel('Latitude')

# Aerosol lowest residual mixture bars
n_mixtures = 9
hist, bin_edges = np.histogram(aerosol_ds.Lowest_Residual_Mixture.data[sub_cond], bins=74, range=(0.5, 74.5))
midpoints = (bin_edges[:-1] + bin_edges[1:])/2
sorted_hist = np.sort(hist)
sorted_hist = sorted_hist[::-1]
sorted_midpoints = midpoints[np.argsort(hist)]
sorted_midpoints = sorted_midpoints[::-1]
ax1.bar(sorted_midpoints.astype('str')[:n_mixtures], sorted_hist[:n_mixtures])
ax1.set_xlabel('Aerosol Mixture')
ax1.set_ylabel('Counts')
ax1.set_title('Lowest Residual Mixture')

# Plot AT3D retreival center & sampling points
# ax00.scatter(target_lon, target_lat, marker='*', c=target_color, s=30, label='Retrieval Center')
# ax01.scatter(target_lon, target_lat, marker='*', c=target_color, s=30, label='Retrieval Center')
plot_box(target_ll, target_ur, f, ax00, c='r')
plot_box(target_ll, target_ur, f, ax01, c='r')
# ax00.scatter(obs.lon17600.data, obs.lat17600, s=3, c='tab:red')
# ax01.scatter(obs.lon17600.data, obs.lat17600, s=3, c='tab:red')

# Adjust window
ax00.set_xlim([ll[0] - lon_pad*lon_range, ur[0] + lon_pad*lon_range])
ax00.set_ylim([ll[1] - lat_pad*lat_range, ur[1] + lat_pad*lat_range])
# ax00.set_xlim(lon_window)
# ax00.set_ylim(lat_window)
ax00.grid()
# ax00.legend(loc='upper left')
# ax01.legend(loc='upper left')
ax00.set_xlabel('Longitude')
ax00.set_ylabel('Latitude')

ax00.set_title('MISR Aerosol Optical Depth')
ax01.set_title('MISR Lowest Residual Mixture')

set_labels([ax00, ax01, ax1])

if(savefigs):
    plt.savefig(
        FIGURE_OUT / '{}_figure05_MISR_aerosol.png'.format(fig_version),
        format='png',
        dpi=300,
        bbox_inches='tight',
        transparent=False
    )


## Figure 6: MISR CTH

In [ ]:
# Subset conditions
lon_range = 0.1
lat_range = 0.1
ll = [target_lon-lon_range, target_lat-lat_range]
ur = [target_lon+lon_range, target_lat+lat_range]
lon_cond = (cth.lon1100.data > ll[0]) & (cth.lon1100.data < ur[0])
lat_cond = (cth.lat1100.data > ll[1]) & (cth.lat1100.data < ur[1])
sub_cond = lon_cond & lat_cond

lon_pad = 1.5
lat_pad = 1.5

f, ax = plt.subplots(1, 2, figsize=(11,4))

# Plot MISR BRF
ax[0].contourf(obs.lon275.data, obs.lat275.data, obs.brf.isel(camera=4).data, vmin=0, vmax=0.20, cmap='gray', levels=100)
ax[0].set_xlim([ll[0] - lon_pad*lon_range, ur[0] + lon_pad*lon_range])
ax[0].set_ylim([ll[1] - lat_pad*lat_range, ur[1] + lat_pad*lat_range])
ax[0].set_xlabel('Longitude')
ax[0].set_ylabel('Latitude')

# Plot MISR CTH
levels = np.arange(0, 1500, 100)
cth_ax = ax[0].contourf(cth.lon1100.data, cth.lat1100.data, cth.cth.data, alpha=1.0, vmin=0, vmax=1400, levels=levels, cmap=cth_cmap)
plt.colorbar(cth_ax, ax=ax[0], label='Cloud Top Height (m)')
ax[0].set_title('MISR 1.1 km Wind Corrected CTH\nOpacity Bias Adjusted')

plot_box(ll, ur, f, ax[0], c='r')

# Plot AT3D retreival center & sampling points
# ax[0].scatter(target_lon, target_lat, marker='*', c=target_color, s=30, label='Retrieval Center')
# ax[0].scatter(obs.lon17600.data, obs.lat17600, s=3, c='tab:red')

bins = np.arange(-1, 1501, 10) + 0.5
h = ax[1].hist(np.ravel(cth.cth.data[sub_cond]), bins=bins)
ax[1].vlines(lcl, 0, np.max(h[0]), colors='tab:orange')
ax[1].set_xlabel('Cloud Top Height (m)')
ax[1].set_ylabel('Count')
ax[1].set_title('MISR 1.1 km Wind Corrected CTH\nOpacity Bias Adjusted')
# print(h[1])

set_labels(ax.flatten())

if(savefigs):
    plt.savefig(
        FIGURE_OUT / '{}_figure06_MISR_CTH.png'.format(fig_version),
        format='png',
        dpi=300,
        bbox_inches='tight',
        transparent=False
    )